# Zaryn Data Engineering | Manufacturing & Distribution Analytics
## Python data curation and transformation notebook

**Purpose:** demonstrate a repeatable Bronze → Silver → Gold workflow for manufacturing and distribution analytics.

- **Bronze:** load raw operational files (or generate synthetic demo data).
- **Silver:** standardize types, clean text, remove duplicates, validate keys, and derive trusted fields.
- **Gold:** create reporting dimensions and facts for Power BI.

> All generated records are synthetic and for portfolio demonstration only. No production or customer data is included.

## 1. Setup

This notebook uses `pandas` and `numpy`. In a local Jupyter environment, install them with:

```bash
pip install pandas numpy
```

The paths are relative to the repository root. If the expected CSV files are not present, the notebook creates a small synthetic dataset so it can run independently.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "sample"
BRONZE_DIR = REPO_ROOT / "data" / "bronze"
SILVER_DIR = REPO_ROOT / "data" / "silver"
GOLD_DIR = REPO_ROOT / "data" / "gold"

for folder in [DATA_DIR, BRONZE_DIR, SILVER_DIR, GOLD_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT.resolve())

## 2. Create or load Bronze data

Expected input files (CSV): `customers.csv`, `products.csv`, `warehouses.csv`, `sales_orders.csv`, `sales_order_lines.csv`, `inventory_transactions.csv`, `shipments.csv`, and `production_orders.csv`.

For a first run, synthetic demo records are generated. To use your own SQL Server exports later, place the matching CSV files in `data/sample/`; the loader will use those files instead.

In [ ]:
def make_demo_data():
    customers = pd.DataFrame([
        (1, "Northstar Retail", "North", "Retail"),
        (2, "Summit Bikes", "West", "Wholesale"),
        (3, "Metro Cycles", "Central", "Retail"),
        (4, "Pioneer Outdoor", "South", "Wholesale"),
    ], columns=["customer_id", "customer_name", "region", "customer_type"])

    products = pd.DataFrame([
        (101, "Mountain Bike", "Bikes", 850.00, "Each"),
        (102, "Road Bike", "Bikes", 1200.00, "Each"),
        (103, "Bike Wheel", "Components", 65.00, "Each"),
        (104, "Bike Chain", "Components", 22.00, "Each"),
        (105, "Handlebar", "Components", 38.00, "Each"),
    ], columns=["product_id", "product_name", "category", "unit_price", "unit_of_measure"])

    warehouses = pd.DataFrame([
        (10, "Dallas Distribution Center", "South"),
        (20, "Austin Warehouse", "South"),
        (30, "Denver Warehouse", "West"),
    ], columns=["warehouse_id", "warehouse_name", "region"])

    orders = pd.DataFrame([
        (5001, 1, "2026-01-05", "Shipped"),
        (5002, 2, "2026-01-18", "Shipped"),
        (5003, 3, "2026-02-03", "Delivered"),
        (5004, 4, "2026-02-19", "Processing"),
        (5005, 1, "2026-03-02", "Delivered"),
        (5006, 2, "2026-03-15", "Shipped"),
    ], columns=["order_id", "customer_id", "order_date", "order_status"])

    order_lines = pd.DataFrame([
        (9001, 5001, 101, 2, 850.00),
        (9002, 5001, 103, 10, 65.00),
        (9003, 5002, 102, 1, 1200.00),
        (9004, 5003, 104, 20, 22.00),
        (9005, 5004, 105, 12, 38.00),
        (9006, 5005, 101, 1, 850.00),
        (9007, 5006, 103, 15, 65.00),
    ], columns=["order_line_id", "order_id", "product_id", "quantity", "unit_price"])

    inventory = pd.DataFrame([
        (1, 101, 10, "2026-01-02", "PURCHASE_RECEIPT", 30, 25500.00),
        (2, 101, 10, "2026-01-05", "SALE_ISSUE", -2, -1700.00),
        (3, 103, 10, "2026-01-05", "SALE_ISSUE", -10, -650.00),
        (4, 102, 30, "2026-01-17", "PURCHASE_RECEIPT", 15, 18000.00),
        (5, 102, 30, "2026-01-18", "SALE_ISSUE", -1, -1200.00),
        (6, 104, 20, "2026-02-02", "PURCHASE_RECEIPT", 50, 1100.00),
        (7, 104, 20, "2026-02-03", "SALE_ISSUE", -20, -440.00),
        (8, 105, 20, "2026-02-18", "PURCHASE_RECEIPT", 25, 950.00),
        (9, 101, 10, "2026-03-01", "SALE_ISSUE", -1, -850.00),
        (10, 103, 10, "2026-03-14", "SALE_ISSUE", -15, -975.00),
    ], columns=["inventory_transaction_id", "product_id", "warehouse_id", "transaction_date",
                "transaction_type", "quantity_change", "inventory_value_change"])

    shipments = pd.DataFrame([
        (7001, 5001, 10, "2026-01-06", "2026-01-09", "Delivered"),
        (7002, 5002, 30, "2026-01-19", "2026-01-23", "Delivered"),
        (7003, 5003, 20, "2026-02-04", "2026-02-06", "Delivered"),
        (7004, 5006, 10, "2026-03-16", None, "In Transit"),
    ], columns=["shipment_id", "order_id", "warehouse_id", "ship_date", "delivery_date", "shipment_status"])

    production = pd.DataFrame([
        (8001, 101, 10, "2026-01-01", "2026-01-04", 25, 24, "Completed"),
        (8002, 102, 30, "2026-01-10", "2026-01-16", 20, 20, "Completed"),
        (8003, 103, 10, "2026-02-01", "2026-02-03", 100, 96, "Completed"),
        (8004, 105, 20, "2026-03-01", None, 40, 18, "In Progress"),
    ], columns=["production_order_id", "product_id", "warehouse_id", "planned_start_date",
                "actual_completion_date", "planned_quantity", "produced_quantity", "production_status"])

    return {
        "customers": customers, "products": products, "warehouses": warehouses,
        "sales_orders": orders, "sales_order_lines": order_lines,
        "inventory_transactions": inventory, "shipments": shipments,
        "production_orders": production
    }

def load_bronze_tables():
    demo = make_demo_data()
    tables = {}
    for name, demo_df in demo.items():
        file_path = DATA_DIR / f"{name}.csv"
        if file_path.exists():
            tables[name] = pd.read_csv(file_path)
            print(f"Loaded source CSV: {file_path}")
        else:
            tables[name] = demo_df.copy()
            print(f"Using synthetic demo data: {name}")
        tables[name]["_ingestion_timestamp"] = datetime.utcnow().isoformat(timespec="seconds")
        tables[name]["_source_system"] = "DEMO_ERP"
        tables[name]["_pipeline_run_id"] = "zaryn-demo-run-001"
        tables[name].to_csv(BRONZE_DIR / f"{name}.csv", index=False)
    return tables

bronze = load_bronze_tables()
bronze["sales_order_lines"].head()

## 3. Silver layer — standardize and curate

The Silver layer applies consistent types and text formatting, removes duplicate business keys (keeping the latest row), and performs basic referential-integrity checks. Rows with invalid relationships are excluded from curated outputs and reported in the validation summary.

In [ ]:
def clean_text_columns(df):
    result = df.copy()
    for col in result.select_dtypes(include=["object"]).columns:
        if not col.startswith("_"):
            result[col] = result[col].astype("string").str.strip()
    return result

silver = {name: clean_text_columns(df) for name, df in bronze.items()}

# Parse dates and numeric columns safely.
date_columns = {
    "sales_orders": ["order_date"],
    "inventory_transactions": ["transaction_date"],
    "shipments": ["ship_date", "delivery_date"],
    "production_orders": ["planned_start_date", "actual_completion_date"],
}
for table, cols in date_columns.items():
    for col in cols:
        silver[table][col] = pd.to_datetime(silver[table][col], errors="coerce")

numeric_columns = {
    "products": ["unit_price"],
    "sales_order_lines": ["quantity", "unit_price"],
    "inventory_transactions": ["quantity_change", "inventory_value_change"],
    "production_orders": ["planned_quantity", "produced_quantity"],
}
for table, cols in numeric_columns.items():
    for col in cols:
        silver[table][col] = pd.to_numeric(silver[table][col], errors="coerce")

# Deduplicate using each table's business key.
primary_keys = {
    "customers": "customer_id", "products": "product_id", "warehouses": "warehouse_id",
    "sales_orders": "order_id", "sales_order_lines": "order_line_id",
    "inventory_transactions": "inventory_transaction_id", "shipments": "shipment_id",
    "production_orders": "production_order_id",
}
for table, key in primary_keys.items():
    silver[table] = silver[table].drop_duplicates(subset=[key], keep="last").reset_index(drop=True)

# Remove invalid mandatory records.
silver["sales_order_lines"] = silver["sales_order_lines"].dropna(
    subset=["order_id", "product_id", "quantity", "unit_price"]
)
silver["inventory_transactions"] = silver["inventory_transactions"].dropna(
    subset=["product_id", "warehouse_id", "transaction_date", "quantity_change"]
)

# Referential integrity filters.
valid_order_ids = set(silver["sales_orders"]["order_id"])
valid_product_ids = set(silver["products"]["product_id"])
valid_customer_ids = set(silver["customers"]["customer_id"])
valid_warehouse_ids = set(silver["warehouses"]["warehouse_id"])

silver["sales_orders"] = silver["sales_orders"][
    silver["sales_orders"]["customer_id"].isin(valid_customer_ids)
].copy()
silver["sales_order_lines"] = silver["sales_order_lines"][
    silver["sales_order_lines"]["order_id"].isin(valid_order_ids)
    & silver["sales_order_lines"]["product_id"].isin(valid_product_ids)
].copy()
silver["inventory_transactions"] = silver["inventory_transactions"][
    silver["inventory_transactions"]["product_id"].isin(valid_product_ids)
    & silver["inventory_transactions"]["warehouse_id"].isin(valid_warehouse_ids)
].copy()
silver["shipments"] = silver["shipments"][
    silver["shipments"]["order_id"].isin(valid_order_ids)
    & silver["shipments"]["warehouse_id"].isin(valid_warehouse_ids)
].copy()
silver["production_orders"] = silver["production_orders"][
    silver["production_orders"]["product_id"].isin(valid_product_ids)
    & silver["production_orders"]["warehouse_id"].isin(valid_warehouse_ids)
].copy()

for name, df in silver.items():
    df.to_csv(SILVER_DIR / f"{name}.csv", index=False)

{ name: len(df) for name, df in silver.items() }

## 4. Data quality checks

These checks provide a compact audit trail for row counts, null business keys, and duplicate business keys. In a production pipeline, persist these results to a monitoring table and fail the pipeline when critical rules are violated.

In [ ]:
quality_rows = []
for table, key in primary_keys.items():
    df = silver[table]
    quality_rows.append({
        "table_name": table,
        "row_count": len(df),
        "null_key_count": int(df[key].isna().sum()),
        "duplicate_key_count": int(df[key].duplicated().sum()),
        "quality_status": "PASS" if df[key].notna().all() and not df[key].duplicated().any() else "CHECK"
    })

data_quality_report = pd.DataFrame(quality_rows)
display(data_quality_report)
data_quality_report.to_csv(SILVER_DIR / "data_quality_report.csv", index=False)

## 5. Gold layer — dimensions and facts for Power BI

The Gold layer uses a star-schema approach. Each fact table has a declared grain. In Power BI, create one-to-many relationships from dimensions to facts and avoid direct fact-to-fact relationships.

In [ ]:
# Date dimension covering the observed order, inventory, and shipment dates.
all_dates = pd.concat([
    silver["sales_orders"]["order_date"],
    silver["inventory_transactions"]["transaction_date"],
    silver["shipments"]["ship_date"],
    silver["shipments"]["delivery_date"],
], ignore_index=True).dropna()

date_range = pd.date_range(all_dates.min().normalize(), all_dates.max().normalize(), freq="D")
dim_date = pd.DataFrame({"date": date_range})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month_number"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["day"] = dim_date["date"].dt.day
dim_date["date"] = dim_date["date"].dt.date

dim_customer = silver["customers"].rename(columns={
    "customer_id": "customer_key", "customer_name": "customer"
}).copy()
dim_product = silver["products"].rename(columns={
    "product_id": "product_key", "product_name": "product"
}).copy()
dim_warehouse = silver["warehouses"].rename(columns={
    "warehouse_id": "warehouse_key", "warehouse_name": "warehouse"
}).copy()

# FactSales grain: one row per sales order line.
fact_sales = (
    silver["sales_order_lines"]
    .merge(silver["sales_orders"][["order_id", "customer_id", "order_date", "order_status"]], on="order_id")
    .merge(silver["products"][["product_id", "category"]], on="product_id")
)
fact_sales["sales_amount"] = fact_sales["quantity"] * fact_sales["unit_price"]
fact_sales["order_date_key"] = pd.to_datetime(fact_sales["order_date"]).dt.strftime("%Y%m%d").astype(int)
fact_sales = fact_sales.rename(columns={
    "customer_id": "customer_key", "product_id": "product_key"
})

# FactInventoryMovement grain: one row per inventory transaction.
fact_inventory = (
    silver["inventory_transactions"]
    .rename(columns={"product_id": "product_key", "warehouse_id": "warehouse_key"})
    .copy()
)
fact_inventory["transaction_date_key"] = pd.to_datetime(
    fact_inventory["transaction_date"]
).dt.strftime("%Y%m%d").astype(int)

# FactProduction grain: one row per production order.
fact_production = (
    silver["production_orders"]
    .rename(columns={"product_id": "product_key", "warehouse_id": "warehouse_key"})
    .copy()
)
fact_production["production_start_date_key"] = pd.to_datetime(
    fact_production["planned_start_date"]
).dt.strftime("%Y%m%d").astype(int)
fact_production["production_variance"] = (
    fact_production["produced_quantity"] - fact_production["planned_quantity"]
)

# FactShipment grain: one row per shipment.
fact_shipment = (
    silver["shipments"]
    .rename(columns={"warehouse_id": "warehouse_key"})
    .merge(silver["sales_orders"][["order_id", "customer_id"]], on="order_id", how="left")
    .copy()
)
fact_shipment["ship_date_key"] = pd.to_datetime(
    fact_shipment["ship_date"]
).dt.strftime("%Y%m%d").astype(int)
fact_shipment["delivery_days"] = (
    pd.to_datetime(fact_shipment["delivery_date"]) - pd.to_datetime(fact_shipment["ship_date"])
).dt.days

gold = {
    "dim_date": dim_date,
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_warehouse": dim_warehouse,
    "fact_sales": fact_sales,
    "fact_inventory_movement": fact_inventory,
    "fact_production": fact_production,
    "fact_shipment": fact_shipment,
}

for name, df in gold.items():
    df.to_csv(GOLD_DIR / f"{name}.csv", index=False)

[(name, df.shape) for name, df in gold.items()]

## 6. Example analytical measures

These are starter calculations for validation and exploration. Recreate business measures in Power BI using DAX after importing the Gold CSV files.

In [ ]:
total_revenue = fact_sales["sales_amount"].sum()
units_sold = fact_sales["quantity"].sum()
completed_shipments = fact_shipment["delivery_date"].notna().sum()
on_time_delivery_pct = (
    (fact_shipment["delivery_days"] <= 3).mean() * 100
    if fact_shipment["delivery_days"].notna().any() else np.nan
)
production_attainment_pct = (
    fact_production["produced_quantity"].sum()
    / fact_production["planned_quantity"].sum() * 100
    if fact_production["planned_quantity"].sum() else np.nan
)

summary = pd.DataFrame([{
    "total_revenue": round(total_revenue, 2),
    "units_sold": int(units_sold),
    "shipments_with_delivery_date": int(completed_shipments),
    "delivery_within_3_days_pct_demo": round(on_time_delivery_pct, 1),
    "production_attainment_pct": round(production_attainment_pct, 1),
}])
display(summary)
summary.to_csv(GOLD_DIR / "portfolio_kpi_summary.csv", index=False)

## 7. Export summary and next steps

Output folders created by this notebook:

- `data/bronze/` — raw landing copies with ingestion metadata.
- `data/silver/` — curated and validated tables.
- `data/gold/` — Power BI-ready dimensions and facts.

### Power BI model guidance

1. Import all CSVs from `data/gold/`.
2. Create one-to-many relationships from `dim_customer`, `dim_product`, `dim_warehouse`, and `dim_date` to the relevant fact tables.
3. Use `fact_sales` for revenue and units sold; `fact_inventory_movement` for stock movement; `fact_production` for production attainment; and `fact_shipment` for delivery performance.
4. Hide technical keys from report users where appropriate.
5. Add DAX measures and report pages for Executive Overview, Inventory & Warehouse, Production, and Distribution.

**Important modeling note:** this starter project uses a simplified date-key pattern and synthetic sample records. A production model should include a full role-playing date strategy (order, ship, delivery, transaction, production dates), a formal inventory valuation method, and documented SLA rules.